In [2]:
import numpy as np 
import matplotlib.pyplot as plt
import scipy.ndimage as ndi         # qui ci sono i metodi per realizzare i filtri
import skimage.io as io 
import skimage.exposure as exp
import os 
os.chdir('../../')
import my_modules.histogramop as hope
import my_modules.my_lib as lib
from skimage.transform import rescale
from skimage.transform import warp
import my_modules.color_cube as cube
import my_modules.color_convertion as conv
from skimage.color import rgb2hsv, hsv2rgb
from sklearn.cluster import k_means


path = 'C:\\Users\\rocco\\Documents\\università\\ESM\\laboratorio\\Immagini\\'

## Tecniche per l'elaborazione

L'elaborazione delle singole componenti delle immagini su scala di grigi non è in generale uguale all'elaborazione vettoriale. L'equivalenza tra le elaborazioni è garantita da due ipotesi:
1. l’algoritmo deve potersi applicare sia a scalari che vettori;
2. l'operazione su ogni componente del vettore deve essere indipendente dalle altre.

Alcune rappresentazioni si prestano meglio di altre per determinate operazioni.

__Osservazioni__:
1. python cerca immagini a colori normalizzate tra 0 e 1.
2. per migliorare il contrasto di un'immagine a colori occorre passare al piano hsv, facendo un elevamento a potenza solo sul terzo canale. L'elevamento a potenza è su base 0 < x < 1.

In [ ]:
# Correzione di Toni e Colore

im = path + 'colori.jpg'
x = np.float32(io.imread(im))
x = x/np.max(x)

# miglioramento del contrasto
y = rgb2hsv(x)
y[:,:,-1] = y[:,:,-1]**2

# in alternativa avremmo dovuto modificare le componenti nel piano RGB singolarmente 

plt.figure(1)
plt.imshow(x, clim=[0,1])
plt.title('originale')
plt.figure(2)
plt.imshow(hsv2rgb(y), clim=[0,1])
plt.title('elaborata')

In [ ]:
# cambio asciugamano figlio della prof

im = path + 'azzurro.jpg'

x = np.float32(io.imread(im))
x = x/np.max(x)
y = rgb2hsv(x)

h,s,v = (y[:,:,0], y[:,:,1], y[:,:,2])

mask = np.int8(((h>0.37) & (h<0.63)) & ((s>0.2)&(s<0.8)) & (v>0.35))
H = (h + 0.40*mask) % 1.0
S = (s + 0.2*s) % 1.0
y[:,:,0] = H
y[:,:,1] = S
z = hsv2rgb(y)

plt.figure(1)
plt.imshow(x, clim=[0,1])
plt.title('originale')
plt.figure(2)
plt.subplot(1,3,1)
plt.imshow(h, clim=[0,1], cmap='gray')
plt.title('tinta')
# plt.colorbar()
plt.subplot(1,3,2)
plt.imshow(s, clim=[0,1], cmap='gray')
plt.title('saturazione')
# plt.colorbar()
plt.subplot(1,3,3)
plt.imshow(v, clim=[0,1], cmap='gray')
plt.title('intensità')

plt.figure(3)
plt.imshow(z, clim=[0,1])
 

## Tecniche class based

Le tecniche class Based si basano sull'elaborazione globale dell'immagine per fornire in output una segmentazione. Gli approcci che vedremo si basano su tecniche di thresholding o clustering per produrre una mappa delle etichette. 
Considerando un'immagine costituita da un oggetto in foreground e uno sfondo, se gli oggetti sono di intensità luminosa e colore uniforme, è possibile eseguire un'operazione di separazione mediante una soglia. Quando la soglia è costante si parla di __thrasholding globale__, e si può determinare osservando l'istogramma dell'immagine. Per il thrasholding locale vedere esercizio_6 di questa esercitazione.
Una procedura automatica per determinare la soglia può essere usata mediante l'algoritmo K-means.

L'algoritmo K-Means può essere applicato anche ad immagini a colori. Il clustering va effettuato sulla base dei vettori che determinano lo spazio di rappresentazione dei pixel dell'immagine. 

In [ ]:
# K-means - applicazione

im = path + 'granelli_riso.tif'

x = np.float32(io.imread(im))

plt.figure(1)
plt.imshow(x, clim=None, cmap='gray')

# k-means vuole in input un vettore, non una matrice. Le relazioni spaziali tra i pixel non hanno importanza. 
d = np.reshape(x, (-1,1))   #vettore colonna, -1 è un jolly

# è analogo a d = np.reshape(x, (N*M, 1))

k = 3 # setto il numero delle classi 

centroid, idx, sum_var = k_means(d,k)

# centroid = vettore contenente il valore medio di ogni classe
# idx = vettore contenente le etichette di ogni pixel 
# sum_var = numero reale: somma delle K varianze calcolate per ogni cluster
print('[MAIN]\tSTATISTICHE:')
print(f'centroid - vettore di {centroid.shape}=')
print(centroid)
print(f'idx - vettore di {idx.shape}=')
print(idx)
print(f'sum_var - float =')
print(sum_var)


y = np.reshape(idx, x.shape)


plt.figure(2)
plt.imshow(y, clim=[0,k-1], cmap='jet')
plt.title('class')
plt.show()

In [ ]:
# K-Means - a colori

im = path + 'Fiori256.bmp'
x = np.float32(io.imread(im))/255   # ricorda di normalizzare le immagini a colore
k = 4
M,N,L = x.shape
d = np.reshape(x, (M*N, L))
centroid, idx, sum_var = k_means(d,k)
y = np.reshape(idx, x.shape[:-1])
print(x.shape[:-1])

plt.figure(1)
plt.imshow(x, clim=None, cmap='gray')

plt.figure(2)
plt.imshow(y, clim=None, cmap='jet')

In [ ]:
# K-Means a colori - parte 2

im = path + 'lenac.jpg'
x = np.float32(io.imread(im))
x = x/np.max(x)

M,N,L = x.shape

k = 4
y = np.reshape(x, (M*N, L))
centroid, idx, sum_var = k_means(y,k)

z = np.reshape(idx, (M,N))

plt.figure(1)
plt.imshow(x, clim=None)
plt.title('input')
plt.figure(2)
plt.imshow(z, clim=None, cmap='jet')
plt.title('label map')

plt.show()